# 🌏 7장 · 끼리끼리 모으기: 군집화 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 7장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 7-1, 7-2, 7-3, 7-4, 7-5, 7-6 |
| ☆ 선택 | 자습이나 심화 | 7-7, 7-8, 7-9, 7-10 |

**데이터**
- 전 세계 규모 5 이상 지진(미국지질조사국 USGS, 1900년~): GitHub에 올라온 CSV(약 21MB)를 실습 7-1이 자동으로 내려받습니다.
- 미니 프로젝트: 서울교통공사 역별 시간대별 승하차 인원(2024년)을 달마다 평균 낸 파일 `서울지하철_역별_시간대별_월평균_승하차_2024.csv`. 책과 함께 나누어 주는 파일을 코랩 왼쪽 파일 창에 올려 두십시오.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다. 실습 7-6은 1~3분쯤 걸립니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch07/data/"
for f in [
    "서울지하철_역별_시간대별_월평균_승하차_2024.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 지진 10만 건 묶어 보기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 지진 데이터는 인터넷 주소에서 바로 읽습니다. 판다스, 넘파이와 함께 기계학습 도구인 사이킷런을 씁니다. 사이킷런의 도구는 모두 `fit( )`으로 데이터를 배우고, 군집 도구는 배운 결과를 `labels_`(군집 번호)에 담아 둡니다. k-평균은 시작점을 무작위로 고르므로 `random_state=42`로 난수의 씨앗을 고정합니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 7-1 ★] 지진 데이터 불러오기와 정리

지진 데이터를 읽어 같은 지진의 중복 기록, 지진이 아닌 기록, 깊이나 규모가 빈 기록을 지우시오. 연대별 기록 수도 그리시오.

① 한글 글꼴을 설정하고 파일을 내려받아 읽습니다. ② 중복과 지진이 아닌 기록을 지웁니다. ③ 1972년과 1973년의 기록을 견주고 연대별 막대그래프를 그립니다.

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

DATA = "Significant_Earthquakes.csv"
URL = ("https://raw.githubusercontent.com/Abdullah-Moataz/Earthquakes/"
       "cf0d033d35f5d45eef6175bab6c60829363d6de4/Significant_Earthquakes.csv")
if not os.path.exists(DATA):                       # 파일이 없을 때만 내려받기(약 21MB)
    urllib.request.urlretrieve(URL, DATA)

df = pd.read_csv(DATA, index_col=0)                # 첫 열은 행 번호
print("처음 읽은 행:", len(df), " 같은 id가 다시 나온 행:", df["id"].duplicated().sum())
df["time"] = pd.to_datetime(df["time"], format="ISO8601")
# 같은 지진이 여러 번 실렸으면 가장 나중에 고친(updated) 기록 하나만 남기고, 다시 시간 순서로
df = df.sort_values("updated").drop_duplicates("id", keep="last").sort_values("time")
print("중복을 지운 뒤:", len(df))
print(df["type"].value_counts().head(4).to_string())

eq = df[(df["type"] == "earthquake")
        & df["latitude"].between(-85, 85) & df["longitude"].between(-180, 180)]
print("지진만, 위치가 맞는 것:", len(eq))
eq = eq.dropna(subset=["depth", "mag"]).copy()      # 깊이·규모가 빈 행 제외
eq["region"] = eq["place"].fillna("unknown").str.split(", ").str[-1]
print("분석할 지진:", len(eq), "건,", eq["time"].min().date(), "~", eq["time"].max().date())
print("규모를 잰 방법:", eq["magType"].nunique(), "가지,",
      eq["magType"].value_counts().head(2).to_dict())

year = eq["time"].dt.year
print("1972년:", (year == 1972).sum(), "건  1973년:", (year == 1973).sum(), "건")
small = eq["mag"] < 5.5                            # 규모 5.5 미만 = 작은 지진
print("작은 지진의 비율(%)  1972년까지:", round(small[year < 1973].mean() * 100, 1),
      " 1973년부터:", round(small[year >= 1973].mean() * 100, 1))

decade = year // 10 * 10                           # 1973 → 1970
eq.groupby(decade)["mag"].size().plot.bar(color="#9a9a9a", figsize=(6, 3))
plt.xlabel("연대")
plt.ylabel("기록된 지진 수")
plt.xticks(rotation=45)
plt.show()

[표 7-1]의 정리 과정을 그대로 따라 102,144건이 남았습니다. 1972년 619건이 1973년 1,362건으로 늘었고, 작은 지진의 비율은 14.3%에서 71.9%로 뛰었습니다. 작은 지진이 그때부터 생긴 것이 아니라, 그때부터 기록되기 시작한 것입니다. `region`은 "101 km ENE of Masohi, Indonesia"처럼 적힌 장소에서 마지막 쉼표 뒤, 곧 나라나 지역 이름만 떼어 낸 것입니다.

> **[코드 읽기] `drop_duplicates("id", keep="last")`**
>
> 같은 id가 여러 줄이면 마지막 줄 하나만 남기라는 뜻입니다. 바로 앞에서 고친 시각(updated) 순으로 줄을 세워 두었으므로, 마지막 줄이 가장 최근에 고친 기록입니다. 중복을 지운 뒤에는 다시 일어난 시각(time) 순으로 세웁니다.

#### [파이썬 실습 7-2 ★] 깊이와 규모 살펴보기

깊이와 규모의 기초 통계를 구하고, 깊이를 세 구간으로 나눈 비율, 10km와 33km로 적힌 비율, 상관계수를 구하시오. 두 분포도 그리시오.

In [ ]:
print(eq[["depth", "mag"]].describe().round(2))
depth_class = pd.cut(eq["depth"], [-10, 70, 300, 800],
                     labels=["얕은(70km 이하)", "중간(70~300km)", "깊은(300km 초과)"])
print((depth_class.value_counts(normalize=True) * 100).round(1).to_string())
d = eq["depth"].round(0)
print("깊이가 정확히 10km:", round((d == 10).mean() * 100, 1), "%,  33km:",
      round((d == 33).mean() * 100, 1), "%")
print("깊이와 규모의 상관계수:", round(eq["depth"].corr(eq["mag"]), 3))
late = eq[year >= 1973]                            # 기록이 고르게 된 뒤
print("1973년부터  규모 6 이상:", (late["mag"] >= 6).sum(), "건, 7 이상:", (late["mag"] >= 7).sum(), "건")
print(pd.cut(eq["depth"], range(200, 701, 50)).value_counts().sort_index().to_string())

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].hist(eq["depth"], bins=range(0, 720, 20), color="#9a9a9a")
ax[0].set_yscale("log")                            # 깊은 쪽의 작은 봉우리가 보이도록
ax[0].set_xlabel("깊이(km)")
ax[0].set_ylabel("지진 수(로그 눈금)")
ax[1].hist(eq["mag"], bins=np.arange(5, 9.6, 0.1), color="#E04B3C")
ax[1].set_xlabel("규모")
plt.tight_layout()
plt.show()

깊이는 평균(62.23)이 가운데 값(33.00)의 두 배 가까이 되는, 오른쪽으로 꼬리가 긴 분포입니다. 표준편차는 깊이가 108.35, 규모가 0.48로 200배 넘게 차이 납니다. 깊이 구간 표에서는 250~300km(435건)와 300~350km(253건)가 골짜기이고, 550~600km(1,294건)에 봉우리가 하나 더 있습니다. 깊이의 최솟값 −3km는 해수면보다 높은 산 아래의 지진을 해수면 기준으로 잰 것입니다.

> **[코드 읽기] `pd.cut(값, 경계)`**
>
> 숫자를 구간으로 나눕니다. 경계가 [−10, 70, 300, 800]이면 "−10 초과 70 이하", "70 초과 300 이하", "300 초과 800 이하"의 세 구간입니다. `value_counts(normalize=True)`는 개수 대신 비율을 냅니다.

#### [파이썬 실습 7-3 ★] k-평균을 손으로 돌려 보기

[예제 7-2]의 점 일곱 개를 넘파이로 두 군집으로 나누되, 배정과 갱신을 직접 코드로 쓰시오.

`pts[:, None, :] - C[None, :, :]`는 점 7개와 중심 2개의 모든 짝에 대해 좌표 차이를 한 번에 구합니다. 2장의 브로드캐스팅입니다. 결과는 7 × 2 × 2 모양이고, 마지막 축을 따라 제곱해 더하면 7 × 2의 거리 제곱 표가 됩니다.

In [ ]:
pts = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5]])
C = pts[[0, 3]].copy()                             # 처음 중심: 점 1과 점 4
for step in range(1, 10):
    d2 = ((pts[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)   # 점-중심 거리의 제곱
    a = d2.argmin(axis=1)                          # ① 배정: 가장 가까운 중심
    print(f"{step}회: 중심 {C.round(2).tolist()}, 배정 {a.tolist()}, "
          f"WCSS {d2.min(axis=1).sum():.2f}")
    newC = np.array([pts[a == j].mean(axis=0) for j in range(2)])  # ② 갱신: 평균
    if np.allclose(newC, C):                       # 중심이 그대로면 멈춤
        break
    C = newC

plt.figure(figsize=(4, 4))
plt.scatter(pts[:, 0], pts[:, 1], c=np.where(a == 1, "#E04B3C", "#9a9a9a"), s=80)
plt.scatter(C[:, 0], C[:, 1], marker="x", s=150, color="black")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

손 계산과 같습니다. 2회째에 세 번째 점 (3, 4)의 배정이 0에서 1로 바뀌고, 3회째에는 중심이 그대로라 멈춥니다. `argmin( )`은 거리가 같으면 앞의 중심을 고르므로, 1회째에 (3, 4)가 첫 번째 군집에 들어갔습니다.

#### [파이썬 실습 7-4 ★] 거리와 표준화

1.3의 지진 세 개로 표준화 전 거리를 구하고, 실제 데이터로 표준화 도구를 만든 뒤 표준화 후 거리를 구하시오. 표준화 없이 k = 3으로 묶은 중심도 구하시오.

In [ ]:
def dist(p, q):                                    # 유클리드 거리
    return np.sqrt(((p - q) ** 2).sum())

q = np.array([[30, 5.0], [40, 8.0], [80, 5.0]])    # [깊이(km), 규모] 지진 1, 2, 3
print("표준화 전  1-2:", round(dist(q[0], q[1]), 2), " 1-3:", round(dist(q[0], q[2]), 2))

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = eq[["depth", "mag"]].to_numpy()
scaler = StandardScaler().fit(X)                   # 평균과 표준편차를 배운다
print("평균:", scaler.mean_.round(2), " 표준편차:", scaler.scale_.round(2))
z = scaler.transform(q)                            # (식 7-3)을 적용한다
print("표준화 후  1-2:", round(dist(z[0], z[1]), 2), " 1-3:", round(dist(z[0], z[2]), 2))
print("깊이 15km, 규모 6.84인 지진의 z:", scaler.transform([[15, 6.84]]).round(2))

raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)   # 표준화 없이 묶으면
for depth, mag in sorted(raw.cluster_centers_.tolist()):
    print(f"표준화 없이 묶은 중심: 깊이 {depth:6.1f}km, 규모 {mag:.2f}")

plt.figure(figsize=(6, 3))
plt.scatter(X[::20, 0], X[::20, 1], c=raw.labels_[::20], cmap="Greys", vmin=-1, s=6)
plt.xlabel("깊이(km)")
plt.ylabel("규모")
plt.show()

표준화 전에는 지진 2가, 뒤에는 지진 3이 지진 1과 가깝습니다. 표준화 없이 묶은 세 중심은 규모가 5.37~5.47로 거의 같습니다. 깊이만 보고 나눈 것입니다. 이 결과로는 규모 9의 대지진과 규모 5의 지진이 같은 군집에 섞입니다.

> **[코드 읽기] `fit( )`, `transform( )`, `inverse_transform( )`**
>
> `StandardScaler( ).fit(X)`는 열마다 평균과 표준편차를 배워 `mean_`과 `scale_`에 담습니다. `transform( )`은 배운 값으로 (식 7-3)을 적용하고, `inverse_transform( )`은 거꾸로 원래 단위로 되돌립니다. 배우기와 적용하기가 나뉘어 있어서, 새 지진이 와도 같은 눈금으로 바꿀 수 있습니다.

#### [파이썬 실습 7-5 ★] k-평균으로 세 유형 찾기

표준화한 깊이와 규모로 k = 3인 k-평균을 돌리고, 군집마다 지진 수, 깊이와 규모의 모습을 구하시오. 시대별 구성비와 정해 넣은 깊이의 비율도 구하시오.

`n_init=10`은 시작점을 바꿔 10번 돌린 뒤 군집 내 제곱합이 가장 작은 결과를 고르라는 뜻입니다. 군집 번호에는 뜻이 없으므로, 중심을 원래 단위로 되돌려 보고 사람이 이름을 붙이는 함수 `name( )`을 만들었습니다.

In [ ]:
Xs = scaler.transform(X)
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Xs)
centers = scaler.inverse_transform(km.cluster_centers_)   # 원래 단위로 되돌리기
print("표준화된 중심:", km.cluster_centers_.round(2).tolist())
print("원래 단위의 중심:", centers.round(2).tolist())

def name(c):                                       # 중심을 보고 사람이 붙이는 이름
    return "C 깊은" if c[0] > 200 else ("B 얕고 강한" if c[1] > 5.7 else "A 얕고 보통")

eq["군집"] = [name(centers[l]) for l in km.labels_]
summary = eq.groupby("군집").agg(지진수=("mag", "size"), 깊이평균=("depth", "mean"),
                                 깊이최소=("depth", "min"), 규모평균=("mag", "mean"),
                                 규모최소=("mag", "min"), 규모최대=("mag", "max"))
summary["비율%"] = (summary["지진수"] / len(eq) * 100).round(1)
print(summary.round(2).to_string())

period = np.where(year < 1973, "1972년까지", "1973년부터")   # 기록이 크게 는 해
print((eq.groupby(period)["군집"].value_counts(normalize=True) * 100).round(1).unstack())
fixed = d.isin([10, 33])                           # 관례로 정해 넣은 깊이
print("정해 넣은 깊이의 비율(%):", (fixed.groupby(eq["군집"]).mean() * 100).round(1).to_dict())

COLOR = {"A 얕고 보통": "#9a9a9a", "B 얕고 강한": "#E04B3C", "C 깊은": "#222222"}
s = eq.sample(3000, random_state=1)
plt.figure(figsize=(6, 3.5))
for g, part in s.groupby("군집"):
    plt.scatter(part["depth"], part["mag"], s=6, color=COLOR[g], label=g)
plt.xlabel("깊이(km)")
plt.ylabel("규모")
plt.legend(markerscale=3)
plt.show()

[표 7-2]의 숫자가 그대로 나옵니다. C의 가장 얕은 지진은 276.2km입니다. 표준화된 중심 (−0.24, 1.42)만 보고는 뜻을 알기 어렵지만, 깊이 35.7km, 규모 6.13으로 되돌리면 "얕고 강한 지진"이라는 이름이 떠오릅니다. 뒤의 두 표는 [표 7-5]의 숫자입니다. 1972년까지의 기록에서는 B가 69.1%로 가장 많고, 1973년부터는 A가 79.2%입니다.

#### [파이썬 실습 7-6 ★] 몇 개로 나눌까, 진짜일까

k를 2부터 8까지 바꿔 가며 군집 내 제곱합과 실루엣 계수를 구하시오. 무리가 없는 두 데이터와도 실루엣을 견주시오.

실루엣은 계산량이 많아 1만 건을 뽑아 계산합니다. `KMeans`는 군집 내 제곱합을 `inertia_`에 담아 둡니다. `rng.uniform( )`은 같은 범위에 점을 고르게 뿌리고, `rng.permutation( )`은 한 열의 순서를 뒤섞습니다. 이 셀은 2~3분 걸릴 수 있습니다.

In [ ]:
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
samp = rng.choice(len(Xs), 10000, replace=False)    # 실루엣은 1만 건 표본으로
rows = []
for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    rows.append([k, m.inertia_, silhouette_score(Xs[samp], m.labels_[samp])])
    if k == 2:
        print("k=2의 두 군집:", sorted(np.bincount(m.labels_).tolist(), reverse=True))
score = pd.DataFrame(rows, columns=["k", "WCSS", "실루엣"])
print(score.round({"WCSS": 0, "실루엣": 3}).to_string(index=False))

# 무리가 없는 데이터와 견주기 (k = 3)
U = rng.uniform(Xs.min(axis=0), Xs.max(axis=0), size=Xs.shape)   # 같은 범위에 고르게 뿌린 점
S = np.column_stack([rng.permutation(Xs[:, 0]),                 # 깊이와 규모를 따로 섞어
                     rng.permutation(Xs[:, 1])])                # 둘의 짝을 끊은 지진
for label, data in [("실제 지진", Xs), ("고르게 뿌린 점", U), ("따로 섞은 지진", S)]:
    m = KMeans(n_clusters=3, n_init=10, random_state=42).fit(data)
    sizes = np.sort(np.bincount(m.labels_)) / len(data) * 100
    print(f"{label:8s} 실루엣 {silhouette_score(data[samp], m.labels_[samp]):.3f}"
          f"  군집 크기(%) {sizes.round(1).tolist()}")

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].plot(score["k"], score["WCSS"], "o-", color="#555555")
ax[0].set_xlabel("k")
ax[0].set_ylabel("WCSS")
ax[1].plot(score["k"], score["실루엣"], "o-", color="#E04B3C")
ax[1].set_xlabel("k")
ax[1].set_ylabel("실루엣 계수")
plt.tight_layout()
plt.show()

[표 7-3]과 [표 7-4]의 숫자입니다. 고르게 뿌린 점도 27.9%, 27.9%, 44.2%로 세 무리가 되었습니다(1.1). 실루엣은 실제 지진 0.585, 고르게 뿌린 점 0.390이고, 따로 섞은 지진은 0.587로 실제와 같습니다. 따로 섞은 지진의 군집 크기도 4.3%, 23.9%, 71.8%로 실제와 같습니다. 세 유형이 각 특징의 분포 모양에서 나왔다는 3.6의 해석을 뒷받침합니다.

#### [파이썬 실습 7-7 ☆] 결정나무로 군집 설명하기

군집 이름을 정답으로 놓고 깊이 2인 결정나무를 학습시켜, 군집을 나누는 규칙과 그 규칙이 맞힌 비율을 구하시오.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(eq[["depth", "mag"]], eq["군집"])
print(export_text(tree, feature_names=["depth", "mag"], decimals=1))
print("군집을 맞힌 비율:", round(tree.score(eq[["depth", "mag"]], eq["군집"]) * 100, 2), "%")

규칙입니다. 결정나무는 표준화하지 않은 원래 단위로 학습시켰습니다. 그래서 "규모 5.7", "깊이 284km"처럼 바로 읽을 수 있는 기준이 나옵니다. 결정나무는 10장에서 자세히 배웁니다.

#### [파이썬 실습 7-8 ☆] 깊이에 로그를 씌우면

깊이에 로그를 씌워 k = 3으로 다시 묶고, 원래 군집과 얼마나 겹치는지 구하시오.

In [ ]:
from sklearn.metrics import adjusted_rand_score

eq["log깊이"] = np.log1p(eq["depth"].clip(lower=0))   # log(1 + 깊이)
XL = StandardScaler().fit_transform(eq[["log깊이", "mag"]])
kl = KMeans(n_clusters=3, n_init=10, random_state=42).fit(XL)
print(eq.groupby(kl.labels_).agg(지진수=("mag", "size"), 깊이중앙값=("depth", "median"),
                                 규모평균=("mag", "mean")).round(2).to_string())
print("원래 군집과 겹치는 정도(조정 랜드 지수):", round(adjusted_rand_score(km.labels_, kl.labels_), 2))

로그를 씌우면 얕은 지진 안의 작은 차이가 커집니다. 그래서 깊이 중앙값 24km 근처의 두 무리와 114km 근처의 무리로 나뉩니다. 두 결과가 얼마나 같은지는 **조정 랜드 지수**(adjusted rand index)로 쟀습니다. 두 군집 결과에서 "같은 무리로 묶인 짝"이 얼마나 일치하는지를 0(우연 수준)에서 1(완전히 같음) 사이로 나타낸 값입니다. 0.49이니 절반쯤만 같습니다. 어느 쪽이 맞는다기보다, 무엇을 보고 싶은지에 따라 특징을 고르는 것이 분석가의 일입니다.

#### [파이썬 실습 7-9 ☆] 계층 군집으로 지역 묶기

지진이 가장 많은 12개 지역의 깊이와 규모 모습을 구하고, 와드 방법의 계층 군집으로 세 무리로 나눈 뒤 덴드로그램을 그리시오.

와드 방법(Ward's method)은 두 무리를 합쳤을 때 군집 내 제곱합이 가장 적게 늘어나는 짝부터 합칩니다. 사이파이(SciPy)의 `linkage( )`가 합치는 순서를 계산하고, `fcluster( )`가 원하는 무리 수에서 나무를 자릅니다.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

top12 = eq["region"].value_counts().head(12).index
sub = eq[eq["region"].isin(top12)]
prof = sub.groupby("region").agg(깊이평균=("depth", "mean"),
                                 중간깊이비율=("depth", lambda s: ((s > 70) & (s <= 300)).mean()),
                                 깊은비율=("depth", lambda s: (s > 300).mean()),
                                 규모평균=("mag", "mean")).loc[top12]
Z = linkage(StandardScaler().fit_transform(prof), method="ward")
prof["무리"] = fcluster(Z, t=3, criterion="maxclust")
print(prof.round(3).sort_values("무리").to_string())
plt.figure(figsize=(6, 4))
dendrogram(Z, labels=prof.index.tolist(), orientation="right", color_threshold=0,
           above_threshold_color="#555555")
plt.xlabel("합칠 때의 거리")
plt.show()

6.2에서 말한 세 무리입니다. 무리 1은 깊이 평균이 60~74km, 무리 3은 중간 깊이 지진의 비율이 8~10%입니다. 네 특징은 단위가 서로 다르므로 여기서도 먼저 표준화했습니다.

#### [파이썬 실습 7-10 ☆] DBSCAN으로 지진 밀집 지대 찾기

규모 6 이상인 지진의 위치로 DBSCAN을 돌려, 밀집 지대의 수와 노이즈 비율, 가장 큰 지대 다섯 곳을 구하시오.

지구는 둥글어서 위도와 경도의 차이로 거리를 재면 북극 쪽으로 갈수록 틀립니다. 그래서 둥근 지구 위의 거리인 하버사인 거리(haversine distance)를 씁니다. 좌표는 라디안으로, 거리 기준 `eps`는 "km ÷ 지구 반지름(6,371km)"으로 넣습니다.

In [ ]:
from sklearn.cluster import DBSCAN

big = eq[eq["mag"] >= 6].copy()                    # 규모 6 이상
rad = np.radians(big[["latitude", "longitude"]])
db = DBSCAN(eps=150 / 6371, min_samples=40,        # 반지름 150km 안에 40개 이상
            metric="haversine", algorithm="ball_tree").fit(rad)
big["zone"] = db.labels_                           # -1은 노이즈
print("규모 6 이상 지진:", len(big), "건")
print("찾은 밀집 지대:", big.loc[big.zone >= 0, "zone"].nunique(), "곳",
      " 노이즈 비율:", round((big.zone == -1).mean() * 100, 1), "%")
size = big[big.zone >= 0].groupby("zone").agg(n=("mag", "size"),
                                              지역=("region", lambda s: s.mode()[0]))
print(size.sort_values("n", ascending=False).head(5).to_string(index=False))
biggest = size["n"].idxmax()                       # 가장 큰 지대에는 어느 지역이?
print(big.loc[big.zone == biggest, "region"].value_counts().head(3).to_dict())

숫자입니다. 가장 큰 지대는 파푸아뉴기니와 바누아투, 솔로몬 제도에 걸쳐 있어, 그 안에서 가장 많이 나온 지역 이름(최빈값)으로 표시했습니다. `eps`와 `min_samples`를 바꾸면 결과가 크게 달라집니다. [연습문제 7-6]에서 확인해 봅니다.

> **핵심 3줄**
>
> k-평균은 가장 가까운 중심에 배정하고 중심을 평균으로 옮기기를 되풀이하며, 거리를 쓰므로 열마다 단위를 맞추는 표준화가 먼저입니다.
>
> 실루엣이 높다고 군집이 진짜라는 뜻은 아니므로, 고르게 뿌린 점이나 열마다 섞은 데이터 같은 기준선과 견주어야 합니다.
>
> 지진을 k = 3으로 나누자 실루엣이 0.585였고, 깊이와 규모를 따로 섞은 데이터도 0.587로 거의 같았습니다.

## 06절의 숫자 확인하기 (선택)

책 06절의 숫자(무작위 시작 10번, 가장 먼 0.5%, 2011년 일본 지진 묶음)를 만든 코드입니다. 실습 7-1~7-10을 먼저 실행한 뒤 돌리십시오.

**S6-1**

In [ ]:
# 6.1 시작점에 따라 달라지는 답 (k = 5, 한 번씩만 돌리기)
runs = [KMeans(n_clusters=5, n_init=1, init="random", random_state=s).fit(Xs).inertia_
        for s in range(10)]
print("무작위 시작 10번의 WCSS:", sorted(np.round(runs).astype(int).tolist()))
print("서로 다른 값:", len(set(np.round(runs))), "가지")
pp = KMeans(n_clusters=5, n_init=1, init="k-means++", random_state=0).fit(Xs).inertia_
print("k-평균++ 한 번:", round(pp))

**S6-2**

In [ ]:
# 6.3 이상 탐지: 가장 가까운 중심까지의 거리가 가장 먼 0.5%
far = np.min(((Xs[:, None, :] - km.cluster_centers_[None]) ** 2).sum(axis=2), axis=1) ** 0.5
eq["중심거리"] = far
top = eq[far >= np.quantile(far, 0.995)].sort_values("중심거리", ascending=False)
print("가장 먼 0.5%:", len(top), "건")
print(top[["time", "mag", "depth", "region"]].head(6).to_string(index=False))

**S6-3**

In [ ]:
# 6.2 여진 묶기: 2011년 일본 주변 규모 5 이상 지진을 위치(100km)와 시간(30일)으로
win = eq[(year == 2011) & eq["latitude"].between(25, 50) & eq["longitude"].between(125, 155)].copy()
main = win.loc[win["mag"].idxmax()]
xk = (win["longitude"] - main["longitude"]) * 111.2 * np.cos(np.radians(38)) / 100
yk = (win["latitude"] - main["latitude"]) * 111.2 / 100
tk = (win["time"] - pd.Timestamp("2011-01-01", tz="UTC")).dt.total_seconds() / 86400 / 30
win["st"] = DBSCAN(eps=0.6, min_samples=10).fit(np.column_stack([xk, yk, tk])).labels_
seq = win[win["st"] == win.loc[main.name, "st"]]
after = seq["time"] - main["time"]
print("2011년 일본 주변:", len(win), "건, 본진", main["time"].date(), "규모", main["mag"])
print("본진과 한 묶음:", len(seq), "건, 첫날", ((after >= pd.Timedelta(0)) & (after < pd.Timedelta(days=1))).sum(),
      "건, 첫 주", ((after >= pd.Timedelta(0)) & (after < pd.Timedelta(days=7))).sum(),
      "건, 한 달 뒤부터", (after >= pd.Timedelta(days=30)).sum(), "건, 전진", (after < pd.Timedelta(0)).sum(), "건")

## [미니 프로젝트 7] 지하철역은 몇 가지 하루를 살까

**질문.** 아침 8시, 어떤 역에서는 사람들이 쏟아져 타고, 어떤 역에서는 쏟아져 내립니다. 서울 지하철역을 시간대별 승하차의 "하루 리듬"만으로 묶으면 몇 가지 유형이 나올까요? 그 유형은 진짜일까요?

**데이터.** 서울교통공사가 공공데이터포털에 올린 "서울교통공사_역별 일별 시간대별 승하차인원 정보" 2024년치입니다. 원자료는 하루, 한 역, 한 호선의 승차(또는 하차)가 한 행이고 199,424행입니다. 이 책에서는 이것을 달마다 평균 낸 파일(서울지하철_역별_시간대별_월평균_승하차_2024.csv)을 함께 나누어 줍니다. 11,506행 × 26열이고, 한 행은 "어느 달, 평일 또는 쉬는 날, 한 역의 승차(또는 하차)를 시간대별로 평균한 값"입니다. 환승역은 호선을 합쳐 역 이름 하나로 셌습니다. 역 이름은 원자료 그대로여서, 4호선은 총신대입구(이수), 7호선은 이수로 적힌 역은 두 역으로 셉니다. 파일을 Colab 왼쪽 파일 창에 끌어다 놓고 시작합니다.


**1단계: 데이터 신원 확인**([파이썬 실습 7-1] 응용). 파일을 읽어 빈칸과 역마다의 줄 수를 세고, 달마다 시간대 합계를 견주시오.

In [ ]:
# 파일이 없으면 업로드 창을 엽니다 (코랩). 책과 함께 받은 CSV 파일을 고르십시오.
import os
if not os.path.exists("서울지하철_역별_시간대별_월평균_승하차_2024.csv"):
    from google.colab import files
    files.upload()

In [ ]:
# 데이터 바꾸기: 다른 해의 파일을 쓸 때는 이 두 줄만 고칩니다.
PATH = "서울지하철_역별_시간대별_월평균_승하차_2024.csv"   # Colab 왼쪽 파일 창에 올린 파일
HOURS = slice("06시이전", "24시이후")                    # 시간대 열의 처음과 끝

sw = pd.read_csv(PATH)
print(sw.shape, "/ 빈칸", sw.isna().sum().sum(), "/ 역", sw["역명"].nunique(), "곳")
per_st = sw.groupby("역명").size()                    # 역마다 몇 줄인가 (12달 × 2 × 2 = 48)
print(per_st.value_counts().to_dict(), per_st[per_st != 48].to_dict())
H = sw.loc[:, HOURS].columns
month = sw[sw["날종류"] == "평일"].groupby("월")[H].sum() / 1000   # 달별 시간대 합(천 명)
print(month[["12-13시", "13-14시", "14-15시"]].round(0).astype(int).T.to_string())

month.T.plot(legend=False, color="#9a9a9a", figsize=(7, 3))
plt.plot(month.loc[4].values, color="#E04B3C", linewidth=2)   # 4월만 빨갛게
plt.xticks(range(len(H)), H, rotation=90)
plt.ylabel("평일 하루 평균(천 명)")
plt.show()

역 241곳 가운데 239곳은 48줄(12달 × 평일·쉬는 날 × 승차·하차)인데 두 곳은 아닙니다. 암사역사공원(22줄)은 2024년 하반기에 새로 연 역이고, 까치울(12줄)은 기록이 일부 달에만 있습니다. 더 큰 문제는 4월입니다. 1장과 3장에서 만난 2024년 4월 13~14시 기록입니다. 13~14시 합계가 4,243천 명으로 다른 달(391~450천 명)의 10배쯤입니다. 원자료를 열어 보면 4월의 13~14시 칸에만 앞뒤 시간대의 몇 배에서 열 배가 넘는 값이 들어간 날이 많습니다. 사람들이 갑자기 그 시간에만 몰렸을 리는 없으니 기록의 잘못으로 보고, 4월은 빼고 씁니다. 13장에서도 이 4월 기록을 "멈추고 기록하기"의 예로 다시 만납니다.

**2단계: 기준선 만들기**([파이썬 실습 7-4], [7-6] 응용). 평일 하루 리듬을 역마다 한 줄로 만드시오. 인원수를 그대로 묶은 결과와, 열마다 따로 섞은 데이터의 실루엣을 기준선으로 구하시오.

In [ ]:
ok = sw[(sw["날종류"] == "평일") & (sw["월"] != 4)]          # 4월은 뺀다
w = ok.assign(**{h: ok[h] * ok["날수"] for h in H})           # 날수만큼 가중
prof = (w.groupby(["역명", "구분"])[H].sum()
        .div(w.groupby(["역명", "구분"])["날수"].sum(), axis=0)
        .unstack("구분"))                                     # 한 역 = 한 줄
prof.columns = [f"{b}_{h}" for h, b in prof.columns]
prof = prof[[f"{b}_{h}" for b in ["승차", "하차"] for h in H]]
total = prof.sum(axis=1)
print("승하차가 0인 역:", total[total == 0].index.tolist())
prof, total = prof[total > 0], total[total > 0]
share = prof.div(total, axis=0) * 100                         # 하루 승하차 가운데 몇 %인가
print(len(share), "개 역 ×", share.shape[1], "칸,  가장 붐비는 역:",
      total.idxmax(), round(total.max()))

# 기준선 1: 인원수를 그대로 묶으면?
k_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit(prof)
print("인원수 그대로 묶은 군집의 평균 하루 승하차:",
      total.groupby(k_raw.labels_).agg(["size", "mean"]).round(0).astype(int).values.tolist())
# 기준선 2: 무리가 없는 데이터 (시간대 열마다 따로 섞기)
Xm = share.to_numpy()
rng2 = np.random.default_rng(0)
Sm = np.column_stack([rng2.permutation(Xm[:, j]) for j in range(Xm.shape[1])])
for k in range(2, 7):
    m1 = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xm)
    m2 = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Sm)
    print(f"k={k}: 실제 실루엣 {silhouette_score(Xm, m1.labels_):.3f}, "
          f"섞은 데이터 {silhouette_score(Sm, m2.labels_):.3f}")

승하차가 0으로만 적힌 역 하나를 빼면 240개 역이 남습니다. 인원수를 그대로 묶으면 하루 평균 2만 5천 명, 6만 7천 명, 11만 8천 명인 역끼리 모입니다. 리듬이 아니라 크기로 나뉜 것입니다. 3.3에서 깊이가 거리를 독차지한 것과 같은 일입니다. 그래서 역마다 "하루 승하차 가운데 그 시간대가 몇 %인가"로 바꾸었습니다. 이렇게 바꾼 데이터의 실루엣은 k = 2에서 0.493, k = 3에서 0.388로, 열마다 섞은 데이터(0.167, 0.120)보다 세 배쯤 높습니다.

**3단계: 유형 찾고 이름 붙이기**([파이썬 실습 7-5] 응용). k = 3으로 묶고, 중심의 하루 리듬을 그려 유형마다 이름을 붙이시오.

In [ ]:
K = 3
kmm = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xm)
cent = pd.DataFrame(kmm.cluster_centers_, columns=share.columns)
am = cent[["승차_07-08시", "승차_08-09시"]].sum(axis=1) - cent[["하차_07-08시", "하차_08-09시"]].sum(axis=1)
print("아침 7~9시 (승차 % − 하차 %):", am.round(1).tolist())
share["유형"] = kmm.labels_
for c in range(K):
    names = share.index[share["유형"] == c]
    top = total[names].sort_values(ascending=False).head(5).index.tolist()
    print(f"유형 {c}: {len(names)}개 역, 붐비는 역 {top}")

fig, ax = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
for c, col in zip(range(K), ["#9a9a9a", "#E04B3C", "#222222"]):
    ax[0].plot(cent.iloc[c, :len(H)].values, color=col, label=f"유형 {c}")
    ax[1].plot(cent.iloc[c, len(H):].values, color=col)
for a_, t in zip(ax, ["승차", "하차"]):
    a_.set_title(t)
    a_.set_xticks([2, 8, 13, 18], ["07시", "13시", "18시", "23시"])
ax[0].set_ylabel("하루 가운데 %")
ax[0].legend(loc="upper center")
plt.show()

실루엣은 k = 2가 가장 높았지만, k = 3을 골랐습니다. 지진에서처럼 쓰임새를 생각했습니다. 유형 0(107개 역)은 아침 7~9시에 타는 사람이 내리는 사람보다 하루의 11.7%p 많습니다. 신림, 수유, 쌍문처럼 사람이 사는 동네의 역입니다. 유형 1(48개 역)은 거꾸로 13.2%p 적습니다. 강남, 서울역, 역삼처럼 일터가 모인 역입니다. 유형 2(85개 역)는 아침의 타고 내림이 거의 같습니다. 잠실, 홍대입구, 고속터미널처럼 사는 사람과 찾아오는 사람이 섞인 역입니다.

**4단계: 군집에 쓰지 않은 정보로 확인하기.** 쉬는 날 하루 승하차를 평일로 나눈 값을 유형마다 견주시오. 일터가 모인 역이라면 쉬는 날에는 한산해야 합니다.

In [ ]:
# 군집에 쓰지 않은 정보로 확인하기: 쉬는 날 승하차 ÷ 평일 승하차
offday = sw[(sw["날종류"] == "쉬는날") & (sw["월"] != 4)]
days = offday.groupby("역명")["날수"].sum() / 2               # 승차·하차 두 줄에 날수가 같이 적혀 있다
off = offday.assign(t=offday[H].sum(axis=1) * offday["날수"]).groupby("역명")["t"].sum() / days
ratio = (off[share.index] / total).rename("쉬는날÷평일")
print(ratio.groupby(share["유형"]).describe()[["count", "25%", "50%", "75%"]].round(2).to_string())

쉬는 날 승하차는 평일의 몇 배일까요? 가운데 값이 유형 1은 0.48, 유형 0은 0.60, 유형 2는 0.64입니다. 쉬는 날 숫자는 군집을 만들 때 쓰지 않았습니다. 그런데도 "일터가 모인 역"은 쉬는 날 평일의 절반 아래로 줄었습니다. 스노가 물 공급 회사로 확인했듯이, 이름이 들어맞는다는 증거 하나를 얻은 것입니다.

**5단계: 한계 쓰기.** 이 유형이 아직 모르는 것을 [표 7-5]처럼 표로 쓰시오. 이 파일에는 서울교통공사가 운영하는 1~8호선 구간만 들어 있습니다. 9호선이나 신분당선만 지나는 역은 없다는 점을 꼭 다루십시오.

**제출물.** 그림 2장(달별 시간대 합계, 유형별 하루 리듬), 숫자 표 1개(k별 실루엣과 섞은 데이터의 실루엣, 유형별 쉬는 날 비율), 결론 5문장 이내.

**[표 7-6] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 4월의 13~14시 값, 줄 수가 다른 역, 0인 역을 찾아 처리 방법을 밝힘 | 빈칸만 확인함 | 점검 없이 바로 묶음 |
| 기준선 비교(20%) | 인원수 그대로 묶은 결과와 섞은 데이터의 실루엣을 모두 견줌 | 하나만 견줌 | 기준선 없이 군집만 씀 |
| 기법 적용(30%) | 비율로 바꾼 까닭을 설명하고 k를 지표와 쓰임새로 고름 | 군집은 맞으나 k를 고른 까닭이 없음 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 쉬는 날 비율로 이름을 확인하고 한계를 표로 셋 이상 씀 | 이름만 붙이고 한계를 한두 가지 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 핵심 숫자는 이렇습니다. 11,506행, 역 241곳(까치울 12줄, 암사역사공원 22줄), 4월 13~14시 합계 4,243천 명(다른 달 391~450천 명)이라 4월을 뺍니다. 승하차가 0인 까치울을 빼면 240개 역입니다. 인원수 그대로 묶으면 하루 2만 5천, 6만 7천, 11만 8천 명처럼 크기로 나뉩니다. 비율로 바꾼 뒤 실루엣은 k = 2가 0.493, k = 3이 0.388이고, 섞은 데이터는 0.167, 0.120입니다. k = 3의 유형은 아침에 타는 동네 역 107곳(아침 승차 − 하차 +11.7%p), 아침에 내리는 일터 역 48곳(−13.2%p), 둘 사이 85곳(+0.3%p)입니다. 쉬는 날 ÷ 평일의 가운데 값은 0.60, 0.48, 0.64로, 군집에 쓰지 않은 정보가 이름을 받쳐 줍니다. 결론의 예: "서울 지하철역의 평일 리듬은 사는 곳, 일하는 곳, 둘이 섞인 곳으로 나뉜다. 일터 역은 쉬는 날 평일의 절반 아래로 줄어 이름이 맞는다. 다만 9호선과 다른 회사 구간이 빠졌고, 2024년 한 해만 보았다."

> **[더 알아보기] 공공데이터포털에서 원자료를 받아 직접 만들기 ☆**
>
> 공공데이터포털(data.go.kr)에서 "서울교통공사 역별 일별 시간대별 승하차인원"을 찾으면 해마다 CSV 파일을 받을 수 있습니다. 받은 파일은 한글이 cp949로 저장되어 있어서 `encoding="cp949"`를 붙여 읽어야 합니다. 해마다 열 이름과 개수가 조금씩 다르니 먼저 `columns`를 찍어 보십시오. 2024년 파일에서 이 책의 파일을 만든 코드는 상자 아래에 실었습니다. 쉬는 날은 토요일, 일요일과 `holidays.KR( )`이 알려 주는 공휴일입니다. 같은 역이 호선마다 따로 적혀 있으므로 역 이름으로 더한 뒤 평균을 냅니다.
>
> 다른 해의 파일로 만들면 1단계의 `PATH` 한 줄만 바꾸어 같은 단계를 되풀이할 수 있습니다.

**[더 알아보기] 코드** (원자료 CSV를 올린 뒤 새 코드 셀에 붙여 넣어 실행하십시오. 실행하면 책의 파일과 똑같은 파일을 새로 만듭니다.)

```python
# [더 알아보기] 공공데이터포털에서 받은 원자료로 이 책의 CSV 파일 만들기
import holidays                                    # 공휴일 목록 (없으면 !pip install holidays)
RAW = "서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv"
raw = pd.read_csv(RAW, encoding="cp949")           # 한글이 cp949로 저장된 파일
print(raw.shape)
hours = raw.columns[6:].tolist()                   # '06시이전' ~ '24시이후'
raw["날짜"] = pd.to_datetime(raw["수송일자"])
kr = holidays.KR(years=2024)
off_day = (raw["날짜"].dt.dayofweek >= 5) | raw["날짜"].dt.date.map(lambda x: x in kr)
raw["날종류"] = np.where(off_day, "쉬는날", "평일")
raw["월"] = raw["날짜"].dt.month
# 환승역은 호선마다 따로 적혀 있으므로 역명으로 더한 뒤, 달·날종류마다 하루 평균
daily = raw.groupby(["역명", "날짜", "월", "날종류", "승하차구분"])[hours].sum().reset_index()
m = daily.groupby(["월", "날종류", "역명", "승하차구분"])[hours].mean().round(1).reset_index()
ndays = daily.groupby(["월", "날종류"])["날짜"].nunique().rename("날수").reset_index()
lines = (raw.groupby("역명")["호선"].unique()
         .map(lambda a: ",".join(sorted(a, key=lambda t: int(t[0])))).rename("호선"))
m = m.merge(ndays, on=["월", "날종류"]).merge(lines, left_on="역명", right_index=True)
m = m.rename(columns={"승하차구분": "구분"})
m.columns = [c.replace("시간대", "시") for c in m.columns]
m = m[["월", "날종류", "날수", "역명", "호선", "구분"] + [h.replace("시간대", "시") for h in hours]]
m.to_csv("서울지하철_역별_시간대별_월평균_승하차_2024.csv", index=False)
print(m.shape)
```

## ✏️ [연습문제 7-6] DBSCAN의 설정 바꾸기

[파이썬 실습 7-10]에서 `eps`를 200km, `min_samples`를 15로 바꿔 다시 돌리고, 밀집 지대의 수, 가장 큰 지대의 지진 수, 노이즈 비율을 구하시오. 결과가 어떻게 달라졌는지 설명하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
db2 = DBSCAN(eps=200 / 6371, min_samples=15,
             metric="haversine", algorithm="ball_tree").fit(rad)
lab = db2.labels_
sizes = pd.Series(lab[lab >= 0]).value_counts()
print("밀집 지대:", len(sizes), "곳, 가장 큰 지대:", sizes.iloc[0], "건,",
      "노이즈:", round((lab == -1).mean() * 100, 1), "%")
print("가장 큰 지대의 지역:", big["region"][lab == sizes.index[0]].value_counts().head(4).to_dict())